# Прогноз согласия клиента на срочный вклад в банковском маркетинге

Датасет: телефонные маркетинговые кампании португальского банка ([Moro et al., 2011](http://hdl.handle.net/1822/14838)).

Файл данных: `bank.arff` — полная выборка (~45 211 наблюдений), упорядочена по дате (май 2008 — ноябрь 2010). Пропусков в данных нет.

### Постановка ML-задачи

**Задача:** бинарная классификация — предсказать, согласится ли клиент на **открытие срочного вклада**.

**Целевая переменная:** `y` — `yes` (согласился) / `no` (отказался).

**Признаки:** 16 переменных о клиенте и параметрах маркетинговой кампании. Признак `duration` исключаем из основной модели: длительность звонка известна только во время контакта и даёт data leakage при планировании обзвона.

**Метрики:** ROC-AUC, PR-AUC (average precision), precision, recall, F1. Accuracy не используем как основную метрику из-за сильного дисбаланса классов (~88% отказов).

**План работы:** загрузка → EDA → предобработка и feature engineering → train / val / test split → обучение моделей → метрики на трёх выборках → борьба с переобучением → выводы.

### Описание колонок

**Данные о клиенте банка**

- **Возраст (`age`)** — число, возраст клиента в годах.
- **Род занятий (`job`)** — категория: `admin.`, `blue-collar`, `entrepreneur`, `housemaid`, `management`, `retired`, `self-employed`, `services`, `student`, `technician`, `unemployed`, `unknown`.
- **Семейное положение (`marital`)** — категория: `married` (женат/замужем), `divorced` (разведён/вдовец), `single` (холост/не замужем).
- **Образование (`education`)** — категория: `primary`, `secondary`, `tertiary`, `unknown`.
- **Просрочка по кредиту (`default`)** — бинарная: `yes` (есть просрочка), `no` (нет).
- **Средний годовой баланс (`balance`)** — число, баланс на счёте в евро (может быть отрицательным).
- **Ипотека (`housing`)** — бинарная: `yes` (есть ипотечный кредит), `no` (нет).
- **Персональный кредит (`loan`)** — бинарная: `yes` (есть), `no` (нет).

**Данные о последнем контакте в текущей кампании**

- **Канал связи (`contact`)** — категория: `cellular`, `telephone`, `unknown`.
- **День месяца (`day`)** — число от 1 до 31, день последнего контакта.
- **Месяц (`month`)** — категория: `jan`, `feb`, `mar`, `apr`, `may`, `jun`, `jul`, `aug`, `sep`, `oct`, `nov`, `dec`.
- **Длительность звонка (`duration`)** — число, длительность последнего контакта в секундах.

**Другие параметры кампании**

- **Число контактов в кампании (`campaign`)** — число, сколько раз клиенту звонили в рамках текущей кампании (включая последний звонок).
- **Дней с прошлого контакта (`pdays`)** — число; `-1` означает, что клиента раньше не контактировали.
- **Контактов до кампании (`previous`)** — число, сколько контактов было с клиентом до текущей кампании.
- **Исход прошлой кампании (`poutcome`)** — категория: `unknown`, `other`, `failure`, `success`.

**Целевая переменная**

- **Согласие на открытие срочного вклада (`y`)** — бинарная: `yes` (клиент согласился), `no` (отказался).

## 1. Загрузка данных

Читаем ARFF через `scipy.io.arff`, переименовываем колонки `V1`…`V16` в осмысленные имена и приводим типы к удобному для pandas виду.

In [15]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.io import arff
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

In [16]:
DATA_PATH = Path('bank.arff')

COLUMN_NAMES = {
    'V1': 'age',
    'V2': 'job',
    'V3': 'marital',
    'V4': 'education',
    'V5': 'default',
    'V6': 'balance',
    'V7': 'housing',
    'V8': 'loan',
    'V9': 'contact',
    'V10': 'day',
    'V11': 'month',
    'V12': 'duration',
    'V13': 'campaign',
    'V14': 'pdays',
    'V15': 'previous',
    'V16': 'poutcome',
    'Class': 'y',
}

CATEGORICAL_COLUMNS = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome', 'y',
]

raw_data, _ = arff.loadarff(DATA_PATH)
df = pd.DataFrame(raw_data)

# В ARFF строковые значения приходят как bytes — декодируем.
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].str.decode('utf-8')

df = df.rename(columns=COLUMN_NAMES)

# Class в ARFF: 1 = no, 2 = yes
df['y'] = df['y'].map({'1': 'no', '2': 'yes'})

for col in CATEGORICAL_COLUMNS:
    df[col] = df[col].astype('category')

INTEGER_COLUMNS = ['age', 'day', 'duration', 'campaign', 'pdays', 'previous']
df[INTEGER_COLUMNS] = df[INTEGER_COLUMNS].astype('int64')
df['balance'] = df['balance'].astype('int64')

In [17]:
print(f'Загружено {df.shape[0]} строк и {df.shape[1]} колонок')
print(f'Доля согласий на срочный вклад: {(df["y"] == "yes").mean():.1%}')

df.head()

Загружено 45211 строк и 17 колонок
Доля согласий на срочный вклад: 11.7%


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   age        45211 non-null  int64   
 1   job        45211 non-null  category
 2   marital    45211 non-null  category
 3   education  45211 non-null  category
 4   default    45211 non-null  category
 5   balance    45211 non-null  int64   
 6   housing    45211 non-null  category
 7   loan       45211 non-null  category
 8   contact    45211 non-null  category
 9   day        45211 non-null  int64   
 10  month      45211 non-null  category
 11  duration   45211 non-null  int64   
 12  campaign   45211 non-null  int64   
 13  pdays      45211 non-null  int64   
 14  previous   45211 non-null  int64   
 15  poutcome   45211 non-null  category
 16  y          45211 non-null  category
dtypes: category(10), int64(7)
memory usage: 2.8 MB


## 2. EDA

Смотрим баланс классов, распределения **всех** признаков и их связь с целевой переменной `y`.
Данные уже упорядочены по времени — это учтём при разбиении на выборки.

План визуализаций (по стилю homework_09 / homework_10):
1. целевая переменная и проверка пропусков/дубликатов;
2. гистограммы числовых признаков;
3. распределения категорий и доля согласий (`yes`) по каждой категории;
4. распределения числовых признаков по классам `y`;
5. корреляции, scatterplot-ы и pairplot;
6. boxplot-ы числовых признаков по `y` (без выбросов, чтобы ось не схлопывалась).

In [ ]:
cat_cols = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

y_counts = df['y'].value_counts()
sns.barplot(x=y_counts.index.astype(str), y=y_counts.values, ax=axes[0], palette='muted')
axes[0].set_title('Распределение целевой переменной')
axes[0].set_xlabel('y')
axes[0].set_ylabel('Количество')

conversion_by_month = df.groupby('month', observed=True)['y'].apply(lambda s: (s == 'yes').mean())
conversion_by_month = conversion_by_month.reindex(month_order)
conversion_by_month.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Доля согласий на вклад по месяцам')
axes[1].set_xlabel('month')
axes[1].set_ylabel('Доля yes')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Пропуски:')
print(df.isna().sum())
print('\nДубликаты:', df.duplicated().sum())
print('\nЧисловые признаки:', num_cols)
print('Категориальные признаки:', cat_cols)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax)
    ax.set_title(col)

axes.ravel()[-1].axis('off')
plt.suptitle('Как распределены числовые признаки', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    order = month_order if col == 'month' else df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, color='steelblue')
    ax.set_title(f'Распределение: {col}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('')

for ax in axes[len(cat_cols):]:
    ax.axis('off')

plt.suptitle('Распределения всех категориальных признаков', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    rates = df.groupby(col, observed=True)['y'].apply(lambda s: (s == 'yes').mean())
    if col == 'month':
        rates = rates.reindex(month_order)
    else:
        rates = rates.sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Доля yes по {col}')
    ax.set_ylabel('Доля yes')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

for ax in axes[len(cat_cols):]:
    ax.axis('off')

plt.suptitle('Связь категориальных признаков с целевой переменной', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
n_cols_grid = 4
n_rows = int(np.ceil(len(num_cols) / n_cols_grid))
fig, axes = plt.subplots(n_rows, n_cols_grid, figsize=(16, n_rows * 3))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    for label, group in df.groupby('y', observed=True):
        ax.hist(group[col], bins=25, alpha=0.55, label=str(label), density=True)
    ax.set_title(col)
    ax.legend()

for ax in axes[len(num_cols):]:
    ax.axis('off')

plt.suptitle('Распределения числовых признаков по классам y', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.3)
plt.title('Матрица корреляций Пирсона (числовые признаки)')
plt.tight_layout()
plt.show()

corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
strong_corr = corr_pairs.reindex(corr_pairs['correlation'].abs().sort_values(ascending=False).index)
strong_corr.head(10)

In [ ]:
top_pairs = strong_corr.head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

sample = df.sample(n=min(4000, len(df)), random_state=RANDOM_STATE)

for ax, (_, row) in zip(axes, top_pairs.iterrows()):
    f1, f2 = row['feature_1'], row['feature_2']
    sns.scatterplot(data=sample, x=f1, y=f2, hue='y', alpha=0.45, s=18, ax=ax)
    ax.set_title(f'{f1} vs {f2}\n(r = {row["correlation"]:.3f})')

plt.suptitle('Попарные scatterplot-ы для наиболее связанных числовых признаков', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pair_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'y']
sample_for_pair = df[pair_cols].sample(n=2000, random_state=RANDOM_STATE)

g = sns.pairplot(
    sample_for_pair,
    hue='y',
    diag_kind='kde',
    corner=True,
    plot_kws={'alpha': 0.35, 's': 15},
)
g.fig.suptitle('Связи между ключевыми числовыми признаками (выборка 2000)', y=1.02)
plt.show()

In [ ]:
# Выбросы растягивают ось (особенно balance, duration, campaign, pdays, previous),
# поэтому для сравнения классов рисуем boxplot без fliers.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    sns.boxplot(
        data=df, x='y', y=col, hue='y', ax=ax,
        palette='Set2', legend=False, showfliers=False,
    )
    ax.set_title(f'Boxplot: {col}')
    ax.set_xlabel('y')
    ax.set_ylabel(col)

axes[-1].axis('off')
plt.suptitle('Числовые признаки по целевой переменной y (без выбросов)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Что видно после EDA

- Классы сильно несбалансированы: отказов (`no`) заметно больше, чем согласий (`yes`).
- Среди категорий сильнее всего с `y` связаны `month`, `poutcome`, `contact`, `job`, `housing`.
- Числовые распределения у классов пересекаются; заметнее всего отличается `duration` (но в модель его не берём из‑за leakage).
- Сильных линейных связей между числовыми признаками мало; заметнее всего пара `pdays`–`previous`.

## 3. Предобработка и feature engineering

**Пропуски:** в датасете их нет — imputation не нужен.

**Специальные коды:**
- `pdays = -1` — клиента раньше не контактировали. Создаём флаг `was_contacted_before` и заменяем `-1` на `0` в числовом признаке.
- категории `unknown` — оставляем как отдельные значения при one-hot кодировании.

**Новые признаки:**
- `was_contacted_before` — был ли контакт в прошлых кампаниях.

**Исключения:**
- `duration` не используем в модели (data leakage).

Для моделирования кодируем target: `yes` → 1, `no` → 0.

In [ ]:
model_df = df.copy()

model_df['was_contacted_before'] = (model_df['pdays'] != -1).astype(int)
model_df.loc[model_df['pdays'] == -1, 'pdays'] = 0
model_df['y_bin'] = (model_df['y'] == 'yes').astype(int)

FEATURE_COLUMNS = [
    'age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan',
    'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome',
    'was_contacted_before',
]
CATEGORICAL_FEATURES = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'poutcome',
]
NUMERIC_FEATURES = [
    'age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'was_contacted_before',
]

model_df[FEATURE_COLUMNS].head()

## 4. Train / validation / test split

Данные упорядочены по дате, поэтому делим **по времени**, а не случайно:
- **train** — первые 60%
- **validation** — следующие 20%
- **test** — последние 20%

Validation используем для подбора гиперпараметров, test — только для финальной оценки.

In [ ]:
n_rows = len(model_df)
train_end = int(n_rows * 0.6)
val_end = int(n_rows * 0.8)

train_df = model_df.iloc[:train_end].copy()
val_df = model_df.iloc[train_end:val_end].copy()
test_df = model_df.iloc[val_end:].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['y_bin']
X_val = val_df[FEATURE_COLUMNS]
y_val = val_df['y_bin']
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df['y_bin']

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(
        f'{name:5} | rows={len(part):5} | '
        f"yes={part['y_bin'].mean():.1%} | month={part['month'].iloc[0]}..{part['month'].iloc[-1]}"
    )

## 5. Моделирование

Сравниваем две модели:
- **Logistic Regression** — интерпретируемый baseline с `class_weight='balanced'`
- **Random Forest** — нелинейная модель, гиперпараметры подбираем на validation через `GridSearchCV`

Preprocessing и модель объединены в `Pipeline`, чтобы не было leakage между train и val/test.

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
            ('num', StandardScaler(), NUMERIC_FEATURES),
        ]
    )


def collect_metrics(model, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y, proba),
        'pr_auc': average_precision_score(y, proba),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
    }


def metrics_table(model, splits, threshold=0.5):
    rows = []
    for split_name, (X_split, y_split) in splits.items():
        metrics = collect_metrics(model, X_split, y_split, threshold=threshold)
        metrics['split'] = split_name
        rows.append(metrics)
    return pd.DataFrame(rows).set_index('split')[['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']]

In [ ]:
log_reg = Pipeline([
    ('pre', make_preprocessor()),
    ('model', LogisticRegression(max_iter=10_000, class_weight='balanced', random_state=RANDOM_STATE)),
])
log_reg.fit(X_train, y_train)

splits = {
    'train': (X_train, y_train),
    'val': (X_val, y_val),
    'test': (X_test, y_test),
}

print('Logistic Regression')
metrics_table(log_reg, splits)

In [ ]:
rf_pipeline = Pipeline([
    ('pre', make_preprocessor()),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid={
        'model__n_estimators': [100, 200],
        'model__max_depth': [8, 12, None],
        'model__min_samples_leaf': [1, 5, 10],
    },
    scoring='average_precision',
    cv=3,
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print('Лучшие параметры Random Forest:', rf_grid.best_params_)
print('\nRandom Forest')
metrics_table(best_rf, splits)

In [ ]:
comparison = pd.concat([
    metrics_table(log_reg, splits).add_prefix('logreg_'),
    metrics_table(best_rf, splits).add_prefix('rf_'),
], axis=1)

comparison

In [ ]:
rf_val_pr_auc = metrics_table(best_rf, {'val': (X_val, y_val)}).loc['val', 'pr_auc']
lr_val_pr_auc = metrics_table(log_reg, {'val': (X_val, y_val)}).loc['val', 'pr_auc']

if rf_val_pr_auc >= lr_val_pr_auc:
    final_model = best_rf
    final_name = 'Random Forest'
else:
    final_model = log_reg
    final_name = 'Logistic Regression'

print(f'Финальная модель: {final_name}')
print(f'PR-AUC на val: {max(rf_val_pr_auc, lr_val_pr_auc):.3f}')

y_test_proba = final_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_test_proba)

plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_test_proba):.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC-кривая на test ({final_name})')
plt.legend()
plt.show()

y_test_pred = (y_test_proba >= 0.5).astype(int)
print(classification_report(y_test, y_test_pred, target_names=['no', 'yes']))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    xticklabels=['Предсказано: no', 'Предсказано: yes'],
    yticklabels=['Факт: no', 'Факт: yes'],
)
plt.xlabel('Предсказание')
plt.ylabel('Факт')
plt.title(f'Confusion matrix на test ({final_name})')
plt.tight_layout()
plt.show()

## 6. Борьба с переобучением

Что применяем:
- **временной split** — test содержит более поздние кампании, модель не «видит будущее»;
- **Pipeline** — preprocessing обучается только на train;
- **регуляризация** — `class_weight='balanced'` в LogReg; ограничение `max_depth` и `min_samples_leaf` в Random Forest;
- **GridSearchCV по PR-AUC** — подбор гиперпараметров по validation, а не по test.

Признак переобучения — заметный разрыв метрик между train и val/test.

In [ ]:
overfit_check = metrics_table(final_model, splits)[['roc_auc', 'pr_auc', 'f1']]
train_test_gap = (
    overfit_check.loc['train', 'roc_auc'] - overfit_check.loc['test', 'roc_auc']
)
print(f'Разрыв ROC-AUC train - test: {train_test_gap:.3f}')
overfit_check

## 7. Выводы

- Построена модель бинарной классификации для предсказания согласия клиента на открытие срочного вклада.
- Данные не требовали заполнения пропусков; основная предобработка — работа с `pdays=-1` и one-hot кодирование категорий.
- Из-за дисбаланса классов ориентируемся на PR-AUC и F1, а не на accuracy.
- Финальную модель выбираем по validation; test используем один раз для итоговой оценки.